# Confusion Matrix & Basic Classification Metrics Lab

A confusion matrix decomposes binary classification predictions into four fundamental counts: True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN). This lab demonstrates how to compute and visualize confusion matrices using scikit-learn, highlights the dangerous **Accuracy Paradox** on imbalanced data, and calculates majority-class baseline benchmarks.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 1. Extracting TP, TN, FP, FN from a Confusion Matrix

Fit a small spam filter dataset (N=20) and extract the four quadrants from scikit-learn's confusion matrix convention (`cm.ravel()`).

In [ ]:
# Ground truth: 1 = Spam, 0 = Not Spam
y_true = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0])
y_pred = np.array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0])

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
total = len(y_true)

print("Scikit-Learn Confusion Matrix (Rows=Actual, Columns=Predicted):")
print(f"                Pred Neg (0)    Pred Pos (1)")
print(f"Actual Neg (0):      {tn:<14}  {fp:<14} (TN, FP)")
print(f"Actual Pos (1):      {fn:<14}  {tp:<14} (FN, TP)")

acc = accuracy_score(y_true, y_pred)
print(f"\nTotal Samples: {total}")
print(f"Correct (TP + TN): {tp + tn}")
print(f"Errors  (FP + FN): {fp + fn}")
print(f"Accuracy:          {acc:.1%}")

## 2. The Accuracy Paradox on Imbalanced Data

Simulate a rare medical condition affecting only 1% of patients (N=1,000). A trivial dummy classifier that always predicts 'Healthy' achieves 99% accuracy while catching zero sick patients.

In [ ]:
# 990 Healthy (0), 10 Sick (1)
y_medical = np.concatenate([np.zeros(990, dtype=int), np.ones(10, dtype=int)])
y_dummy = np.zeros(1000, dtype=int)  # Always predict 0

cm_med = confusion_matrix(y_medical, y_dummy)
tn_m, fp_m, fn_m, tp_m = cm_med.ravel()

print(f"Trivial Model Accuracy: {accuracy_score(y_medical, y_dummy):.1%}")
print(f"True Positives Caught:  {tp_m} / 10")
print(f"False Negatives Missed: {fn_m} / 10")
print(f"Sensitivity / Recall:   {tp_m / (tp_m + fn_m):.1%}")
print("\nVerdict: The model boasts 99% accuracy but has 0% utility!")

## 3. Benchmarking Against Majority-Class Baselines

Always compare model accuracy against the zero-rule majority baseline: $\max(P, N) / \text{Total}$.

In [ ]:
n_neg, n_pos = 120, 75
y_exam = np.concatenate([np.zeros(n_neg, dtype=int), np.ones(n_pos, dtype=int)])

# Imperfect model
np.random.seed(42)
y_model = y_exam.copy()
flip_indices = np.random.choice(len(y_exam), size=32, replace=False)
y_model[flip_indices] = 1 - y_model[flip_indices]

acc_model = accuracy_score(y_exam, y_model)
baseline_acc = max(n_neg, n_pos) / len(y_exam)

print(f"Total Samples:       {len(y_exam)}")
print(f"Majority Baseline:   {baseline_acc:.1%} (predicting only Negative)")
print(f"Trained Model Acc:   {acc_model:.1%}")
print(f"Genuine Improvement: {acc_model - baseline_acc:+.1%} over naive guessing")